Parsing CSV and Excel Files - Structured Data

In [1]:
import pandas as pd
import os

In [2]:
os.makedirs("data/structured_files", exist_ok=True)

In [ ]:
# Create Sample Data
data = {
    "Product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam"],
    "Category": ["Electronics", "Electronics", "Electronics", "Electronics", "Electronics"],
    "Price": [999.99, 25.99, 49.99, 199.99, 89.99],
    "Stock": [10, 100, 50, 20, 30],
    "Description": [
        "High-performance laptop with 16GB RAM and 512GB SSD.",
        "Wireless mouse with ergonomic design.",
        "Mechanical keyboard with RGB lighting.",
        "27-inch 4K UHD monitor with HDR support.",
        "1080p HD webcam with built-in microphone."
    ]
}

# Save as csv
df = pd.DataFrame(data)
df.to_csv("data/structured_files/products.csv", index=False)
print("Sample CSV file created at data/structured_files/products.csv")

In [6]:
# Save as Excel with Multiple Sheets
with pd.ExcelWriter("data/structured_files/inventory.xlsx") as writer:
    df.to_excel(writer, sheet_name="Products", index=False)
    
    summary_data = {
        "Category": ["Electronics", "Accessories"],
        "Total_Items": [5, 0],
        "Total_Value": [1365.95, 100.00]
    }

    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)
print("Sample CSV and Excel files created successfully in 'data/structured_files' directory.")

Sample CSV and Excel files created successfully in 'data/structured_files' directory.


CSV Processing

In [7]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

e:\Ultimate RAG Bootcamp Using Langchain, LangGraph & Langsmith\RAGUDEMY\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Method 1: Using CSVLoader - Each row becomes a separate document
print("Method 1: Using CSVLoader")
print("CSV Loader - Row-Based Documents")

csv_loader = CSVLoader(
    file_path="data/structured_files/products.csv",
    encoding="utf-8",
    csv_args={"delimiter": ",", "quotechar": '"'}
)

csv_docs = csv_loader.load()
print(csv_docs)
print(f"Number of documents loaded: {len(csv_docs)}")
print(f"First document content:\n{csv_docs[0].page_content}")
print(f"Metadata of first document:\n{csv_docs[0].metadata}")
print(f"Content of first document:\n{csv_docs[0].page_content}")

Method 1: Using CSVLoader
CSV Loader - Row-Based Documents
[Document(metadata={'source': 'data/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 10\nDescription: High-performance laptop with 16GB RAM and 512GB SSD.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Electronics\nPrice: 25.99\nStock: 100\nDescription: Wireless mouse with ergonomic design.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Electronics\nPrice: 49.99\nStock: 50\nDescription: Mechanical keyboard with RGB lighting.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 199.99\nStock: 20\nDescription: 27-inch 4K UHD monitor with HDR support.'), Document(metadata={'source': 'data/structured_files/products.csv', 'row'

In [13]:
# Method 2: Custom CSV Processing - Each row becomes a separate document with custom formatting

from typing import List
from langchain_core.documents import Document

def process_csv_intelligently(file_path: str) -> List[Document]:
    df = pd.read_csv(file_path)
    documents = []
    for _, row in df.iterrows():
        content = f"Product: {row['Product']}\nCategory: {row['Category']}\nPrice: ${row['Price']}\nStock: {row['Stock']}\nDescription: {row['Description']}"
        metadata = {"source": file_path, "row_index": _, "product": row["Product"], "category": row["Category"]}
        
        doc = Document(page_content=content, metadata=metadata)
        documents.append(doc)

    return documents


process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product': 'Laptop', 'category': 'Electronics'}, page_content='Product: Laptop\nCategory: Electronics\nPrice: $999.99\nStock: 10\nDescription: High-performance laptop with 16GB RAM and 512GB SSD.'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product': 'Mouse', 'category': 'Electronics'}, page_content='Product: Mouse\nCategory: Electronics\nPrice: $25.99\nStock: 100\nDescription: Wireless mouse with ergonomic design.'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product': 'Keyboard', 'category': 'Electronics'}, page_content='Product: Keyboard\nCategory: Electronics\nPrice: $49.99\nStock: 50\nDescription: Mechanical keyboard with RGB lighting.'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 3, 'product': 'Monitor', 'category': 'Electronics'}, page_content='Product: Monitor\nCategory: Ele

In [14]:
"""
CSV Processing Strategies:

1. Row-Based Documents (CSVLoader):
   - Each row in the CSV file is treated as a separate document.
    - Metadata includes the source file and row index.
    - Content is the raw CSV row data.
2. Custom CSV Processing:
   - Each row is processed to create a more human-readable document.
    - Metadata includes additional context like product name and category.
    - Content is formatted to highlight key information about each product.
3. Column-Based Documents:
   - Each column could be treated as a separate document, especially if columns represent distinct entities (e.g., product descriptions).
4. Summary Document:
   - Create a single document that summarizes the entire CSV file, such as an overview of all products and their categories.    

"""

'\nCSV Processing Strategies:\n\n1. Row-Based Documents (CSVLoader):\n   - Each row in the CSV file is treated as a separate document.\n    - Metadata includes the source file and row index.\n    - Content is the raw CSV row data.\n2. Custom CSV Processing:\n   - Each row is processed to create a more human-readable document.\n    - Metadata includes additional context like product name and category.\n    - Content is formatted to highlight key information about each product.\n3. Column-Based Documents:\n   - Each column could be treated as a separate document, especially if columns represent distinct entities (e.g., product descriptions).\n4. Summary Document:\n   - Create a single document that summarizes the entire CSV file, such as an overview of all products and their categories.    \n\n'

Excel Processing

In [16]:
# Method 1: Using Pandas for full Control
print("Pandas based Excel Processing")
def process_excel_with_pandas(file_path: str) -> List[Document]:
    xls = pd.ExcelFile(file_path)
    documents = []
    
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name)
        
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Number of rows: {len(df)}\n"
        sheet_content += df.to_string(index=False)

        doc = Document(page_content=sheet_content, metadata={"source": file_path, "sheet_name": sheet_name,
         "num_rows": len(df), "num_columns": len(df.columns), "data_types": df.dtypes.to_dict()})
        documents.append(doc)   
    return documents

Pandas based Excel Processing


In [17]:
excel_docs = process_excel_with_pandas("data/structured_files/inventory.xlsx")
print(f"Number of documents loaded: {len(excel_docs)}")

Number of documents loaded: 2


In [18]:
excel_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_types': {'Product': <StringDtype(storage='python', na_value=nan)>, 'Category': <StringDtype(storage='python', na_value=nan)>, 'Price': dtype('float64'), 'Stock': dtype('int64'), 'Description': <StringDtype(storage='python', na_value=nan)>}}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nNumber of rows: 5\n Product    Category  Price  Stock                                          Description\n  Laptop Electronics 999.99     10 High-performance laptop with 16GB RAM and 512GB SSD.\n   Mouse Electronics  25.99    100                Wireless mouse with ergonomic design.\nKeyboard Electronics  49.99     50               Mechanical keyboard with RGB lighting.\n Monitor Electronics 199.99     20             27-inch 4K UHD monitor with HDR support.\n  Webcam Electronics  89.99     30            1080p HD webcam with built-

In [23]:
# UnstructuredExcelLoader - Each sheet becomes a separate document with raw content
from langchain_community.document_loaders import UnstructuredExcelLoader

print("UnstructuredExcelLoader - Sheet-Based Documents")
unstructured_excel_loader = UnstructuredExcelLoader(
    file_path="data/structured_files/inventory.xlsx",
    encoding="utf-8",
    mode="elements"
)
unstructured_excel_docs = unstructured_excel_loader.load()
print(f"Number of documents loaded: {len(unstructured_excel_docs)}")
print(unstructured_excel_docs[0])

UnstructuredExcelLoader - Sheet-Based Documents
Number of documents loaded: 2
page_content='Product Category Price Stock Description Laptop Electronics 999.99 10 High-performance laptop with 16GB RAM and 512GB SSD. Mouse Electronics 25.99 100 Wireless mouse with ergonomic design. Keyboard Electronics 49.99 50 Mechanical keyboard with RGB lighting. Monitor Electronics 199.99 20 27-inch 4K UHD monitor with HDR support. Webcam Electronics 89.99 30 1080p HD webcam with built-in microphone.' metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-03-14T21:09:21', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>10</td><td>High-performance laptop with 16GB RAM and 512GB SSD.</td></tr><tr><td>Mouse</td><td>Electronics</td><td>25.99